# FREUID — Data Setup (VESSL Workspace)

Downloads competition data into `/root/repo/data/` using `kagglehub`.

**Prerequisites:**
- Run inside the `freuid-hy` VESSL workspace Jupyter server
- Have your Kaggle API key ready (`kaggle.json` or username + key)

Jupyter URL: `https://jupyter-wsp-3io23arqffzt.betelgeuse.cloud.vessl.ai/?token=2f3nruoehnchn0ti`

In [ ]:
# Install kagglehub if not present
!pip install -q kagglehub

In [ ]:
import os

# Set Kaggle credentials — paste your values here
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'
os.environ['KAGGLE_KEY']      = 'YOUR_KAGGLE_KEY'

In [ ]:
import kagglehub

path = kagglehub.competition_download('the-freuid-challenge-2026-ijcai-ecai')
print('Downloaded to:', path)

In [ ]:
import shutil
from pathlib import Path

src  = Path(path)
dest = Path('/root/repo/data')
dest.mkdir(parents=True, exist_ok=True)

# Copy everything kagglehub downloaded into data/
for item in src.iterdir():
    target = dest / item.name
    if target.exists():
        print(f'skip (exists): {target}')
        continue
    if item.is_dir():
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)
    print(f'copied: {item.name}')

print('\ndata/ contents:')
for p in sorted(dest.rglob('*')):
    if p.is_file() and p.suffix not in ('.jpeg', '.jpg', '.png'):
        print(' ', p.relative_to(dest))

In [ ]:
# Sanity check — count images and verify expected CSV files exist
import pandas as pd

train_images = list((dest / 'train' / 'train').glob('*.jpeg'))
test_images  = list((dest / 'public_test' / 'public_test').glob('*.jpeg'))
labels       = pd.read_csv(dest / 'train_labels.csv')

print(f'Train images : {len(train_images)}')
print(f'Test images  : {len(test_images)}')
print(f'Label rows   : {len(labels)}')
print(f'Label cols   : {list(labels.columns)}')
print(f'Fraud rate   : {labels.label.mean():.3f}')
print(f'Doc types    : {labels.type.value_counts().to_dict()}')